# Chicago City Data — SQL Analysis

**IBM SQL for Data Science with Python | Coursera | Final Assignment**

This notebook analyses three real-world Chicago datasets using SQL and Python:
- **Crime data** — 533 reported incidents across the city
- **Census data** — socioeconomic indicators across 78 community areas
- **Public schools data** — performance metrics for 566 schools

The analysis answers 10 questions about crime patterns, school safety, and community hardship — demonstrating SQL skills including filtering, aggregation, JOINs, and subqueries.

## Setup — Connect to database and load datasets

In [ ]:
import csv, sqlite3, pandas

# Create database connection
con = sqlite3.connect("FinalDB.db")
cur = con.cursor()
print("Connected to FinalDB.db")

In [ ]:
!pip install ipython-sql prettytable --quiet
import prettytable
prettytable.DEFAULT = 'DEFAULT'
%load_ext sql
%sql sqlite:///FinalDB.db

In [ ]:
# Load all three CSV datasets into the database
df = pandas.read_csv("ChicagoCensusData.csv")
df.to_sql("CENSUS_DATA", con, if_exists='replace', index=False, method="multi")
print(f"Census data loaded: {len(df)} rows")

df = pandas.read_csv("ChicagoCrimeData.csv")
df.to_sql("CHICAGO_CRIME_DATA", con, if_exists='replace', index=False, method="multi")
print(f"Crime data loaded: {len(df)} rows")

df = pandas.read_csv("ChicagoPublicSchools.csv")
df.to_sql("CHICAGO_PUBLIC_SCHOOLS_DATA", con, if_exists='replace', index=False, method="multi")
print(f"Schools data loaded: {len(df)} rows")

In [ ]:
# Confirm all three tables exist in the database
%sql SELECT name FROM sqlite_master WHERE type='table'

---
## Question 1 — Total number of crimes recorded

**Question:** What is the total number of crimes recorded in the CRIME table?

**SQL concepts used:** `COUNT()`

In [ ]:
%sql SELECT COUNT(*) AS TOTAL_CRIMES FROM CHICAGO_CRIME_DATA

**Answer:** There are **533** crimes recorded in the dataset.

---
## Question 2 — Community areas with low per capita income

**Question:** List community areas (name and number) with per capita income less than 11,000.

**SQL concepts used:** `SELECT`, `WHERE`

In [ ]:
%sql SELECT COMMUNITY_AREA_NAME, COMMUNITY_AREA_NUMBER \
     FROM CENSUS_DATA \
     WHERE PER_CAPITA_INCOME < 11000

**Answer:** 4 community areas have a per capita income below $11,000: West Garfield Park, South Lawndale, Fuller Park, and Riverdale — all on the south and west sides of Chicago.

---
## Question 3 — Crimes involving a minor

**Question:** List all case numbers for crimes involving a minor. How many rows are retrieved?

**SQL concepts used:** `SELECT`, `WHERE`, `LIKE`

In [ ]:
%sql SELECT CASE_NUMBER, PRIMARY_TYPE, DESCRIPTION \
     FROM CHICAGO_CRIME_DATA \
     WHERE DESCRIPTION LIKE '%MINOR%'

**Answer:** **2 rows** are retrieved. The `LIKE '%MINOR%'` pattern matches any description containing the word MINOR anywhere in the text.

---
## Question 4 — Kidnapping crimes involving a child

**Question:** Identify all kidnapping crimes involving a child.

**SQL concepts used:** `WHERE` with multiple conditions using `AND`, `LIKE`

In [ ]:
%sql SELECT CASE_NUMBER, PRIMARY_TYPE, DESCRIPTION, LOCATION_DESCRIPTION, ARREST \
     FROM CHICAGO_CRIME_DATA \
     WHERE PRIMARY_TYPE = 'KIDNAPPING' \
     AND DESCRIPTION LIKE '%CHILD%'

**Answer:** 1 kidnapping crime involving a child is recorded — described as CHILD ABDUCTION/STRANGER. The query uses `AND` to filter on both crime type and description simultaneously.

---
## Question 5 — Unique types of crimes recorded in schools

**Question:** What are the unique types of crimes recorded in schools?

**SQL concepts used:** `SELECT DISTINCT`, `WHERE`, `LIKE`

In [ ]:
%sql SELECT DISTINCT PRIMARY_TYPE \
     FROM CHICAGO_CRIME_DATA \
     WHERE LOCATION_DESCRIPTION LIKE '%SCHOOL%' \
     ORDER BY PRIMARY_TYPE

**Answer:** 6 unique crime types occur in schools: ASSAULT, BATTERY, CRIMINAL DAMAGE, CRIMINAL TRESPASS, NARCOTICS, and PUBLIC PEACE VIOLATION. 

The two key clauses used are **`SELECT DISTINCT`** (to return unique values only) and **`WHERE`** (to filter by location).

---
## Question 6 — Average safety score for middle schools

**Question:** What was the average safety score for middle schools?

**SQL concepts used:** `AVG()`, `WHERE`

In [ ]:
%sql SELECT AVG(SAFETY_SCORE) AS AVG_SAFETY_SCORE \
     FROM CHICAGO_PUBLIC_SCHOOLS_DATA \
     WHERE "Elementary, Middle, or High School" = 'MS'

**Answer:** The average safety score for middle schools (MS) is **48.0**. Note the column name requires double quotes because it contains commas and spaces.

---
## Question 7 — Five community areas with highest poverty rate

**Question:** List the 5 community areas with the highest percentage of households below the poverty line.

**SQL concepts used:** `ORDER BY`, `LIMIT`

In [ ]:
%sql SELECT COMMUNITY_AREA_NAME, PERCENT_HOUSEHOLDS_BELOW_POVERTY \
     FROM CENSUS_DATA \
     ORDER BY PERCENT_HOUSEHOLDS_BELOW_POVERTY DESC \
     LIMIT 5

**Answer:** The 5 most impoverished community areas are Riverdale, Fuller Park, Englewood, North Lawndale, and East Garfield Park.

The clauses added to the query are **`ORDER BY PERCENT_HOUSEHOLDS_BELOW_POVERTY DESC`** and **`LIMIT 5`**.

---
## Question 8 — Most crime-prone community area

**Question:** Which community area number has the most criminal incidents?

**SQL concepts used:** `COUNT()`, `GROUP BY`, `ORDER BY`, `LIMIT`

In [ ]:
%sql SELECT COMMUNITY_AREA_NUMBER, COUNT(*) AS CRIME_COUNT \
     FROM CHICAGO_CRIME_DATA \
     GROUP BY COMMUNITY_AREA_NUMBER \
     ORDER BY CRIME_COUNT DESC \
     LIMIT 5

**Answer:** Community area number **25** has the most criminal incidents with 43 recorded crimes.

---
## Question 9 — Community area with highest hardship index (subquery)

**Question:** Use a subquery to find the name of the community area with the highest hardship index.

**SQL concepts used:** Subquery, `MAX()`, `WHERE`

In [ ]:
%sql SELECT COMMUNITY_AREA_NAME, HARDSHIP_INDEX \
     FROM CENSUS_DATA \
     WHERE HARDSHIP_INDEX = (SELECT MAX(HARDSHIP_INDEX) FROM CENSUS_DATA)

**Answer:** **Riverdale** has the highest hardship index of 98 — the maximum possible on the scale. The subquery `(SELECT MAX(HARDSHIP_INDEX) FROM CENSUS_DATA)` runs first, finds the value 98, then the outer query uses that value to retrieve the community name.

---
## Question 10 — Community with the most crimes (JOIN)

**Question:** What is the name of the community with the most number of crimes?

**SQL concepts used:** `JOIN`, `COUNT()`, `GROUP BY`, `ORDER BY`, subquery

In [ ]:
%sql SELECT c.COMMUNITY_AREA_NAME, COUNT(*) AS CRIME_COUNT \
     FROM CHICAGO_CRIME_DATA cr \
     JOIN CENSUS_DATA c ON cr.COMMUNITY_AREA_NUMBER = c.COMMUNITY_AREA_NUMBER \
     GROUP BY c.COMMUNITY_AREA_NAME \
     ORDER BY CRIME_COUNT DESC \
     LIMIT 1

**Answer:** **Austin** is the community area with the most crimes, recording 43 incidents. This query joins the crime table with the census table on `COMMUNITY_AREA_NUMBER` — the shared key — to translate the area number into a readable name.

---
## Summary

This analysis of Chicago's crime, census, and school datasets demonstrated the following SQL techniques:

| Technique | Used in |
|---|---|
| `COUNT()`, `AVG()`, `MAX()` | Q1, Q6, Q8 |
| `WHERE` with `LIKE` | Q3, Q4, Q5 |
| `SELECT DISTINCT` | Q5 |
| `GROUP BY` + `ORDER BY` + `LIMIT` | Q7, Q8, Q10 |
| Subquery | Q9 |
| `JOIN` across two tables | Q10 |

**Key findings:**
- Austin (community area 25) is the most crime-prone neighbourhood with 43 incidents
- Riverdale has both the highest hardship index (98) and is among the lowest per capita income areas (<$11,000)
- Middle schools average a safety score of only 48 out of 99
- Only 2 crimes involving minors are recorded, and 1 child kidnapping incident
- 6 distinct crime types occur in school settings, with battery being the most common category

---
*Giovani Saut | IBM SQL for Data Science with Python | Coursera Plus*